# VidyaAI: Fine-tuning Gemma 4 E4B with QLoRA

Fine-tune Gemma 4 E4B for multilingual Indian education.

**Runtime:** Colab Pro — **L4 GPU + High-RAM**

In [ ]:
%%capture
!pip install --upgrade transformers accelerate peft bitsandbytes trl
!pip install datasets huggingface_hub sentencepiece protobuf

In [ ]:
# Login to HuggingFace — paste your token when prompted
# Get token from: https://huggingface.co/settings/tokens
from huggingface_hub import notebook_login
notebook_login()

## 1. Load Model

In [ ]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_NAME = "google/gemma-4-e4b"
MAX_SEQ_LENGTH = 512  # Short enough for L4, our QA pairs are < 400 tokens anyway

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
)

model.config.use_cache = False
print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")

import gc; gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory used: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

## 2. Add LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Gemma 4 uses Gemma4ClippableLinear wrappers that PEFT doesn't recognize.
# Unwrap them to expose the inner Linear4bit layers.
def unwrap_clippable_linear(model):
    for name, module in list(model.named_modules()):
        if module.__class__.__name__ == "Gemma4ClippableLinear":
            parent_name = ".".join(name.split(".")[:-1])
            child_name = name.split(".")[-1]
            parent = model.get_submodule(parent_name) if parent_name else model
            setattr(parent, child_name, module.linear)
    return model

model = unwrap_clippable_linear(model)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Attention only — saves ~3GB
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after LoRA: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

## 3. Prepare Dataset

In [ ]:
from datasets import load_dataset, Dataset
import json

HINDI_SYSTEM = "तुम VidyaAI हो, एक बुद्धिमान शिक्षा सहायक। NCERT पाठ्यक्रम के अनुसार सरल हिंदी में जवाब दो। उदाहरणों का उपयोग करो।"
ENGLISH_SYSTEM = "You are VidyaAI, an intelligent education assistant. Answer according to NCERT curriculum in clear, simple English with examples."

def format_to_gemma(instruction, input_text, output_text):
    return (
        f"<start_of_turn>system\n{instruction}<end_of_turn>\n"
        f"<start_of_turn>user\n{input_text}<end_of_turn>\n"
        f"<start_of_turn>model\n{output_text}<end_of_turn>"
    )

# ── Source 1: Hindi Alpaca-style instruction data ──
print("Loading Hindi instruction data...")
hindi_ds = load_dataset("ravithejads/samvaad-hi-en", split="train[:3000]", trust_remote_code=True)
hindi_examples = []
for row in hindi_ds:
    src = row.get("src", "").strip()
    tgt = row.get("tgt", "").strip()
    if src and tgt:
        hindi_examples.append({"text": format_to_gemma(HINDI_SYSTEM, src, tgt)})
print(f"  Hindi instruction: {len(hindi_examples)} examples")

# ── Source 2: English educational QA (SciQ — science questions) ──
print("Loading SciQ...")
sciq = load_dataset("allenai/sciq", split="train[:3000]")
sciq_examples = []
for row in sciq:
    q = row.get("question", "").strip()
    a = row.get("correct_answer", "").strip()
    support = (row.get("support", "") or "")[:300].strip()
    if q and a:
        full_answer = f"{a}\n\nExplanation: {support}" if support else a
        sciq_examples.append({"text": format_to_gemma(ENGLISH_SYSTEM, q, full_answer)})
print(f"  SciQ: {len(sciq_examples)} examples")

# ── Source 3: English QA (OpenBookQA) ──
print("Loading OpenBookQA...")
obqa = load_dataset("allenai/openbookqa", "main", split="train")
obqa_examples = []
for row in obqa:
    q = row.get("question_stem", "").strip()
    choices = row.get("choices", {})
    answer_key = row.get("answerKey", "")
    if q and choices and answer_key:
        labels = choices.get("label", [])
        texts = choices.get("text", [])
        if answer_key in labels:
            idx = labels.index(answer_key)
            a = texts[idx]
            all_choices = "\n".join([f"{l}) {t}" for l, t in zip(labels, texts)])
            full_answer = f"Answer: {answer_key}) {a}\n\nAll options:\n{all_choices}"
            obqa_examples.append({"text": format_to_gemma(ENGLISH_SYSTEM, q, full_answer)})
print(f"  OpenBookQA: {len(obqa_examples)} examples")

# ── Source 4: Hand-crafted NCERT seed examples (high quality) ──
SEED = [
    (HINDI_SYSTEM, "प्रकाश संश्लेषण क्या है?",
     "प्रकाश संश्लेषण वह प्रक्रिया है जिसमें हरे पौधे सूर्य के प्रकाश की ऊर्जा का उपयोग करके कार्बन डाइऑक्साइड और पानी से ग्लूकोज बनाते हैं। इस प्रक्रिया में ऑक्सीजन भी निकलती है।\n\nसमीकरण: 6CO₂ + 6H₂O + सूर्य का प्रकाश → C₆H₁₂O₆ + 6O₂\n\nयह प्रक्रिया पत्तियों में मौजूद क्लोरोफिल की मदद से होती है।"),
    (HINDI_SYSTEM, "पाइथागोरस प्रमेय क्या है? उदाहरण के साथ समझाइए।",
     "पाइथागोरस प्रमेय कहता है कि एक समकोण त्रिभुज में, कर्ण का वर्ग अन्य दो भुजाओं के वर्गों के योग के बराबर होता है।\n\nसूत्र: c² = a² + b²\n\nउदाहरण: अगर दो भुजाएं 3 cm और 4 cm हैं:\nc² = 3² + 4² = 9 + 16 = 25\nc = √25 = 5 cm"),
    (HINDI_SYSTEM, "न्यूटन के गति के नियम बताइए।",
     "न्यूटन के तीन नियम:\n\n1. जड़त्व का नियम: वस्तु अपनी अवस्था में रहती है जब तक बाहरी बल न लगे। उदाहरण: बस रुकने पर आगे गिरना।\n\n2. F = ma: बल = द्रव्यमान × त्वरण। उदाहरण: भारी गेंद को फेंकने में ज़्यादा बल लगता है।\n\n3. क्रिया-प्रतिक्रिया: हर क्रिया की बराबर और विपरीत प्रतिक्रिया होती है। उदाहरण: रॉकेट का उड़ना।"),
    (HINDI_SYSTEM, "भारत की आज़ादी कब और कैसे मिली?",
     "भारत को 15 अगस्त 1947 को ब्रिटिश शासन से आज़ादी मिली।\n\nमुख्य बिंदु:\n1. महात्मा गांधी ने अहिंसा और सत्याग्रह के माध्यम से स्वतंत्रता आंदोलन का नेतृत्व किया।\n2. 1942 में 'भारत छोड़ो आंदोलन' ने ब्रिटिश शासन को हिला दिया।\n3. सुभाष चंद्र बोस ने आज़ाद हिंद फ़ौज बनाई।\n4. अंततः माउंटबेटन योजना के तहत भारत और पाकिस्तान का विभाजन हुआ।"),
    (HINDI_SYSTEM, "कोशिका क्या है? इसके प्रकार बताइए।",
     "कोशिका जीवन की सबसे छोटी इकाई है।\n\nदो मुख्य प्रकार:\n1. प्रोकैरियोटिक कोशिका: केंद्रक नहीं होता। उदाहरण: बैक्टीरिया।\n2. यूकैरियोटिक कोशिका: केंद्रक होता है। उदाहरण: पौधे और जानवरों की कोशिकाएं।"),
    (HINDI_SYSTEM, "अम्ल और क्षार में क्या अंतर है?",
     "अम्ल: स्वाद खट्टा, pH < 7, नीले लिटमस को लाल करता है। उदाहरण: नींबू, सिरका।\nक्षार: स्वाद कड़वा, pH > 7, लाल लिटमस को नीला करता है। उदाहरण: साबुन, बेकिंग सोडा।\n\nउदासीनीकरण: अम्ल + क्षार → लवण + जल"),
    (HINDI_SYSTEM, "भारत के मौलिक अधिकार क्या हैं?",
     "भारतीय संविधान में 6 मौलिक अधिकार हैं:\n1. समानता का अधिकार (अनुच्छेद 14-18)\n2. स्वतंत्रता का अधिकार (अनुच्छेद 19-22)\n3. शोषण के विरुद्ध अधिकार (अनुच्छेद 23-24)\n4. धार्मिक स्वतंत्रता का अधिकार (अनुच्छेद 25-28)\n5. सांस्कृतिक और शैक्षिक अधिकार (अनुच्छेद 29-30)\n6. संवैधानिक उपचारों का अधिकार (अनुच्छेद 32)"),
    (HINDI_SYSTEM, "सौरमंडल में कितने ग्रह हैं?",
     "8 ग्रह: बुध, शुक्र, पृथ्वी, मंगल, बृहस्पति, शनि, अरुण, वरुण। प्लूटो 2006 से बौना ग्रह है।"),
    (ENGLISH_SYSTEM, "What is the water cycle?",
     "The water cycle: Evaporation → Condensation → Precipitation → Collection. Sun heats water, it rises as vapor, forms clouds, falls as rain, flows back to oceans. Repeats endlessly."),
    (ENGLISH_SYSTEM, "Explain the concept of democracy.",
     "Democracy: government by the people through free elections.\nKey features: universal suffrage, rule of law, fundamental rights, independent judiciary.\nIndia is the world's largest democracy since 1950."),
    (ENGLISH_SYSTEM, "What is photosynthesis?",
     "Photosynthesis: plants make glucose from CO₂ + H₂O using sunlight.\n6CO₂ + 6H₂O + Sunlight → C₆H₁₂O₆ + 6O₂\nHappens in chlorophyll-containing leaves. Oxygen is released as byproduct."),
    (ENGLISH_SYSTEM, "Explain Newton's three laws of motion.",
     "1. Inertia: objects stay at rest/motion unless forced. Example: bus stop lurch.\n2. F=ma: force = mass × acceleration. Example: harder hit = faster ball.\n3. Action-reaction: equal opposite forces. Example: rocket propulsion."),
    (ENGLISH_SYSTEM, "Difference between mitosis and meiosis?",
     "Mitosis: 2 identical cells, diploid, for growth. One division.\nMeiosis: 4 different cells, haploid, for gametes. Two divisions + crossing over.\nHumans: mitosis=46 chromosomes, meiosis=23."),
]

seed_examples = [{"text": format_to_gemma(s, i, o)} for s, i, o in SEED]
print(f"  Seed examples: {len(seed_examples)}")

# ── Combine all sources ──
all_examples = seed_examples + hindi_examples + sciq_examples + obqa_examples
dataset = Dataset.from_list(all_examples).shuffle(seed=42)

print(f"\nTotal dataset: {len(dataset)} examples")
print(f"Sample:\n{dataset[0]['text'][:300]}...")

## 4. Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,  # Effective batch size = 8
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=10,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
        report_to="none",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        save_strategy="epoch",
        dataloader_pin_memory=False,
    ),
)

In [ ]:
# Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_mem / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Memory reserved: {start_gpu_memory} GB / {max_memory} GB")

In [ ]:
# Train!
trainer_stats = trainer.train()
print(f"Training completed in {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## 5. Test the Fine-tuned Model

In [ ]:
model.eval()

test_questions = [
    "प्रकाश संश्लेषण क्या है?",
    "भारत के पहले प्रधानमंत्री कौन थे?",
    "What is the formula for area of a circle?",
    "गुरुत्वाकर्षण बल क्या है?",
]

for q in test_questions:
    prompt = (
        "<start_of_turn>system\n"
        "तुम VidyaAI हो, एक बुद्धिमान शिक्षा सहायक। NCERT पाठ्यक्रम के अनुसार सरल हिंदी में जवाब दो।"
        "<end_of_turn>\n"
        f"<start_of_turn>user\n{q}<end_of_turn>\n"
        "<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"A: {response}")

## 6. Save & Upload to HuggingFace

In [ ]:
# Save LoRA adapter locally
model.save_pretrained("vidyaai-gemma4-lora")
tokenizer.save_pretrained("vidyaai-gemma4-lora")
print("Saved LoRA adapter to vidyaai-gemma4-lora/")

In [ ]:
# Upload to HuggingFace Hub
# First: huggingface-cli login
HF_REPO = "yashkuceriya/vidyaai-gemma4-lora"  # Change to your HF username

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"Uploaded to https://huggingface.co/{HF_REPO}")